In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from ultralytics import YOLO

# 1. โหลดโมเดลมาเพื่อเริ่มต้นเทรน (ใช้ yolov11n.pt ถูกต้องแล้วสำหรับการเริ่ม Fine-tune)
model = YOLO("yolo11n.pt")

# 2. เริ่มเทรนโมเดลด้วย Dataset ของคุณ
model.train(
    data="/content/drive/MyDrive/Monkey_Dataset/data.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    lr0=0.001,
    device="0"
)

# 3. ประเมินโมเดล (ระบบจะใช้ตัวที่ดีที่สุดให้อัตโนมัติ หรือจะระบุ path ไปที่ best.pt ก็ได้ครับ)
metrics = model.val()


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Monkey_Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscr

In [ ]:
from ultralytics import YOLO
import cv2
from matplotlib import pyplot as plt

# โหลดโมเดล best.pt
model = YOLO('best.pt')

# ทดสอบภาพ (เปลี่ยน path ภาพตามที่คุณต้องการทดสอบ)
results = model('/content/Human.jpg')

# กำหนด threshold ความมั่นใจขั้นต่ำ
threshold = 0.7

for result in results:
    # กรองกล่องเฉพาะที่มั่นใจ >= threshold
    boxes = result.boxes
    filtered_boxes = [box for box in boxes if float(box.conf) >= threshold]

    if not filtered_boxes:
        print("ไม่พบวัตถุที่มีความมั่นใจเพียงพอ แต่จะแสดงภาพต้นฉบับครับ")
        # ถ้าไม่มีกล่องที่ผ่านเกณฑ์ จะเคลียร์ boxes ให้เป็นค่าว่าง เพื่อให้แสดงภาพเปล่าๆ
        result.boxes = None
    else:
        # หากมีกล่องที่ผ่านเกณฑ์ ให้กรองเฉพาะกล่องนั้น
        result.boxes = boxes[boxes.conf >= threshold]

    # วาดผลลัพธ์ (ถ้าไม่มีกล่อง จะแสดงภาพเฉยๆ)
    img_with_boxes = result.plot()

    # แสดงภาพใน Colab
    img_rgb = cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title("Detection Result (Fish YOLO)")
    plt.show()

In [5]:
# Export Model to ONNX
# ==============================

# โหลดโมเดลที่ดีที่สุดจาก training
best_model = YOLO('/content/drive/MyDrive/Monkey_Dataset/best.pt')

# Export เป็น ONNX
onnx_model = best_model.export(
    format="onnx",
    imgsz=640,
    opset=12,
    simplify=True
)

print("ONNX model saved:", onnx_model)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 101 layers, 2,597,537 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Monkey_Dataset/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 184ms
Prepared 4 packages in 568ms
Installe